# AIDisclose quick start

This notebook runs the full pipeline on the **synthetic** demo corpus bundled with the package (fictional companies). It needs no network access. Replace the source with `EdgarSource`, `XbrlFilingsSource` or your own folder to analyse real reports.

In [ ]:
from importlib import resources
from pathlib import Path

import aidisclose as aid

print(aid.__version__)
data = Path(str(resources.files("aidisclose.demo").joinpath("data")))

## 1. Source and pipeline

In [ ]:
src = aid.LocalSource(data, metadata=data / "metadata.csv")
filings = src.list_filings()
results = aid.Pipeline(backend="rules").run(filings, source=src)
results.documents[["firm_name", "fiscal_year", "status", "n_baseline_sentences", "n_substantive", "n_weak"]]

## 2. Sentence labels and the cues behind them

In [ ]:
results.sentences[["firm_name", "fiscal_year", "section", "state", "reason", "text"]].head(12)

In [ ]:
import json

row = results.sentences.iloc[0]
print(row["text"])
print(json.dumps(json.loads(row["details"]), indent=2))

## 3. Firm-year and year-level indicators

In [ ]:
panel = aid.firm_year_panel(results)
yearly = aid.group_panel(panel, by=["fiscal_year"])
yearly[["fiscal_year", "n_firms", "share_keyword", "share_claim", "share_substantive", "gap_keyword_substantive_pp"]]

## 4. Evidence trail for one indicator value

In [ ]:
results.evidence(firm_id="harbor", fiscal_year=2024)[["section", "reason", "text"]]

## 5. Real data

```python
src = aid.EdgarSource(user_agent="Jane Doe jane.doe@university.edu")
filings = src.list_filings(tickers=["MSFT"], start_year=2019, end_year=2024)
results = aid.Pipeline(sections=["10-K:1", "10-K:1A", "10-K:7"]).run(filings, source=src)
```